# Phase 12 — RDX-Zeuge v2

v1 lief, war aber ungültig: eine Zufallsstichprobe aus dem Katalog kippte in
3 von 120 Fällen (nur **14** eindeutige Prompts tragen
`language-switching-english`), und der Quervergleich wich um 2.24e-01 ab —
verursacht durch `torch.cdist`, das ab 25 Zeilen in float32 über den
Gram-Trick rechnet und bei Residuen Ränge kippt.

v2 misst den kontinuierlichen **Fremdschrift-Druck** statt der Kipprate,
erzwingt alle Ziel-Prompts in den Pool, rechnet Abstände stabil in float64 und
prüft die Portierung zusätzlich präzisionsgleich gegen den Originalcode.

Selbstversorgend — **frische Runtime**, dann nur diese Zelle. ~15 min.

In [ ]:
# === RDX-ZEUGE v2: zwei Defekte aus v1 behoben =============================
# Was v1 gezeigt hat und was daran kaputt war:
#   (1) Zielgroesse ohne Spanne. 120 zufaellige Katalog-Prompts kippten in
#       3/120 Faellen, mittlere Kipprate 0.006. Grund: der Katalog enthaelt
#       ueber 7000 Transkripte zu vielen Verhalten, aber nur 14 EINDEUTIGE
#       Prompts mit behavior_id="language-switching-english". Eine Zufalls-
#       stichprobe trifft das Zielverhalten praktisch nie, und ohne Varianz
#       in der Zielgroesse ist jedes eta^2 Rauschen.
#       -> v2 misst statt der Kipprate den kontinuierlichen FREMDSCHRIFT-DRUCK
#          an den ersten vier Antwortpositionen (gierige Dekodierung). Der hat
#          keinen Boden und streut auch dort, wo nie wirklich gekippt wird.
#          Zusaetzlich werden alle Prompts des Zielverhaltens in den Pool
#          gezwungen, und die behavior_id dient als Familienetikett.
#   (2) Quervergleich wich um 2.24e-01 ab. Kein Formelfehler: torch.cdist
#       rechnet ab 25 Zeilen per Default ueber den Gram-Trick in float32.
#       Residuen haben eine grosse gemeinsame Komponente (Norm >> Streuung),
#       da loescht sich x^2+y^2-2xy Stellen aus und einzelne Raenge kippen.
#       -> v2 rechnet die Analyse mit stabilen paarweisen Abstaenden in
#          float64 und fuehrt den Quervergleich ZUSAETZLICH mit einer
#          praezisionsgleichen Nachbildung. Dort muss die Abweichung null
#          sein; die verbleibende Differenz zur stabilen Rechnung ist der
#          Praezisionsverlust des Upstream-Pfades, nicht unser Fehler.
#
# Unveraendert die eigentliche Frage: ein Zeuge, den nicht das Modell schreibt,
# sondern den wir aus der Repraesentationsdifferenz ablesen, kann nicht
# abgekoppelt werden. Damit er verschwindet, muss die Differenz verschwinden -
# und das IST die Verhaltensaenderung. Kosten(eliminieren) = Kosten(anpassen).
#
# Kontrolle bleibt k-means je Einzelschicht; zusaetzlich wird der Druck NUR
# innerhalb der Verhaltensfamilien gemischt (Spezifitaetstest). Das Verdikt
# haengt am globalen p, der Familientest wird getrennt berichtet - bei nur 14
# Ziel-Prompts sind Familie und Druck fast kollinear, ein strenges Gate darauf
# wuerde den Befund per Konstruktion erwuergen.
#
# Vorregistrierung Nr. 27: KEIN-ZEUGE ~40%, NICHT-DIFFERENZIELL ~35%,
# RDX-ZEUGE ~25%. Selbstversorgend, FRISCHE Runtime, einzige Zelle. ~15 min.

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, numpy as np
# ---------------- Selbstversorgung: Modell + Prompts sicherstellen ----------
import glob, json, gc
for _n in ("model_b","tok_b"):                  # Base-Reste aus Cell 28 raus
    if _n in globals():
        try: del globals()[_n]
        except Exception: pass
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError(("GPU nicht leer genug (%.1f GB frei, ~45 noetig): vermutlich "
        "belegt noch ein frueheres Modell den Speicher. Loesung: Laufzeit -> "
        "Sitzung neu starten, dann NUR diese Zelle ausfuehren.")%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)

import matplotlib.pyplot as plt
N_PROMPTS=120; K_SAMP=4; MAX_NEW=16; NPOS=4; MAXCHARS=1200; SEED=0
L_EARLY=3; L_LATE=23
BETA=5.0; GAMMA_SCALE=25.0; N_CLUSTERS=5; FILTER_THRESH=1.1; ADD_NULL=True; N_PERM=2000
MASK_NPZ=(glob.glob("/content/drive/MyDrive/**/vocab_foreign_masks.npz",recursive=True) or [""])[0]
META_JL=(glob.glob("/content/drive/MyDrive/**/weird_meta.jsonl",recursive=True) or [""])[0]
SCAFF="<|im_start|>user\n"
def think_prefix(u,th=""):
    return SCAFF+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n"+th+"\n</think>\n\n"
# ---------- RDX-Kern, portiert aus Erikiss/RDX @ a301a499 src/rdx.py --------
# construct_graph(sim_function="neighborhood") + apply_diff_function
# ("locally_biased") + cluster_graph("spectral") + post_process.
# numpy statt torch, damit die Logik ohne GPU testbar ist; identische Formeln.
def rdx_dist(X,mode="stabil"):
    """Euklidische Abstaende. "stabil" = paarweise Differenzen in float64.
       "torch32" = Gram-Trick in float32, also das, was torch.cdist bei mehr als
       25 Zeilen per Default rechnet. Residuen haben einen grossen gemeinsamen
       Anteil (Norm >> Streuung), da verliert der Gram-Trick Stellen und kippt
       Raenge - deshalb rechnen wir die Analyse stabil und benutzen "torch32"
       nur, um den Quervergleich mit dem Upstream bei GLEICHER Arithmetik zu
       fuehren."""
    if mode=="torch32":
        X=np.asarray(X,dtype=np.float32); sq=(X*X).sum(1)
        D=np.sqrt(np.maximum(sq[:,None]+sq[None,:]-2.0*(X@X.T),0.0))
    else:
        from scipy.spatial.distance import cdist
        X=np.asarray(X,dtype=np.float64); D=cdist(X,X)
    D=np.asarray(D,dtype=np.float64); np.fill_diagonal(D,0.0); return D
def rdx_rank_dm(X,mode="stabil"):
    """Abstaende -> doppeltes argsort = Rangmatrix (skalenfrei), upstream Z. 178-182"""
    return np.argsort(np.argsort(rdx_dist(X,mode),axis=1),axis=1).astype(np.float64)
def rdx_diff(r1_dm,r0_dm,gamma,symmetrize="mean"):
    """upstream apply_diff_function, Zweig 'locally_biased'"""
    if symmetrize=="mean":
        r0_dm=(r0_dm+r0_dm.T)/2.0; r1_dm=(r1_dm+r1_dm.T)/2.0
    elif symmetrize=="max":
        r0_dm=np.maximum(r0_dm,r0_dm.T); r1_dm=np.maximum(r1_dm,r1_dm.T)
    denom=np.minimum(r1_dm,r0_dm)+1.0
    d10=np.tanh(gamma*(r1_dm-r0_dm)/denom)
    d01=np.tanh(gamma*(r0_dm-r1_dm)/denom)
    return d10,d01
def rdx_graph(X0,X1,beta=5.0,gamma=None,gamma_scale=25.0,symmetrize="mean",mode="stabil"):
    """gibt die vier Matrizen zurueck, die upstream construct_graph liefert.
       Richtung '10': hohe Affinitaet = in X1 nah, in X0 fern -> vom spaeten
       Layer NEU gruppiert. Richtung '01': umgekehrt (auseinandergezogen)."""
    n=X0.shape[0]
    if gamma is None: gamma=gamma_scale/float(n)
    r0=rdx_rank_dm(X0,mode); r1=rdx_rank_dm(X1,mode)
    r0_am=np.exp(-beta*r0); r1_am=np.exp(-beta*r1)
    d10,d01=rdx_diff(r1,r0,gamma,symmetrize)
    return dict(am_10=np.exp(-beta*d10),am_01=np.exp(-beta*d01),
                r0_am=r0_am,r1_am=r1_am,diff_10=d10,diff_01=d01,gamma=gamma,
                r0_rank=r0,r1_rank=r1)
def rdx_post_process(labels,am,thresh=-1.0):
    """upstream post_process: Cluster nach mittlerer Innen-Affinitaet sortieren,
       alles unter thresh in den Null-Cluster 0."""
    labels=np.asarray(labels); means=[]
    uniq=np.unique(labels)
    for li in uniq:
        m=labels==li
        means.append(float(am[np.ix_(m,m)].mean()))
    means=np.array(means); order=np.argsort(means)
    new=np.zeros_like(labels)
    for i,ci in enumerate(order):
        new[labels==uniq[ci]]=0 if means[ci]<thresh else i+1
    return new,dict(zip(uniq.tolist(),means.tolist()))
def rdx_cluster(am,n_clusters,thresh=-1.0,seed=0,add_null_cluster=True):
    """upstream cluster_graph('spectral') inkl. Symmetrisierung und post_process"""
    from sklearn.cluster import SpectralClustering
    A=am.copy()
    if not np.allclose(A,A.T): A=(A+A.T)/2.0
    k=n_clusters+int(add_null_cluster)
    cl=SpectralClustering(n_clusters=k,affinity="precomputed",random_state=seed,
                          eigen_solver="arpack",assign_labels="kmeans",n_init=10)
    lab=cl.fit_predict(np.maximum(A,0.0))
    return rdx_post_process(lab,am,thresh)
def kmeans_labels(X,k,seed=0):
    """Kontrolle 'nur eine Schicht': k-means direkt auf der Repraesentation.
       Bewusst NICHT spektral auf exp(-beta*Rang) - diese Matrix ist praktisch
       diagonal (exp(-5)=0.007) und die Clusterung darauf springt zwischen
       Laeufen; k-means ist stabil und als Baseline fair."""
    from sklearn.cluster import KMeans
    Xs=(X-X.mean(0))/ (X.std(0)+1e-9)
    return KMeans(n_clusters=k,random_state=seed,n_init=10).fit_predict(Xs)+1
# ---------- Auswertung: erklaert die Partition das Kippverhalten? -----------
def eta_squared(labels,y):
    """Anteil der Varianz der Kipprate, den die Partition erklaert"""
    y=np.asarray(y,dtype=np.float64); labels=np.asarray(labels)
    tot=((y-y.mean())**2).sum()
    if tot<=0: return 0.0
    bet=0.0
    for li in np.unique(labels):
        m=labels==li
        bet+=m.sum()*(y[m].mean()-y.mean())**2
    return float(bet/tot)
def perm_p_within(labels,y,fam,n_perm=2000,seed=0):
    """Permutation NUR innerhalb der Familien - so kann Familienstruktur
       (gleiche Prompt-Herkunft) keine Signifikanz erzeugen."""
    rng=np.random.default_rng(seed)
    y=np.asarray(y,dtype=np.float64); fam=np.asarray(fam)
    groups=[np.where(fam==f)[0] for f in np.unique(fam)]
    obs=eta_squared(labels,y); ge=0
    for _ in range(n_perm):
        yp=y.copy()
        for g in groups:
            if len(g)>1: yp[g]=y[rng.permutation(g)]
        if eta_squared(labels,yp)>=obs: ge+=1
    return obs,(1.0+ge)/(1.0+n_perm)
def perm_p(labels,y,n_perm=2000,seed=0):
    """Permutationstest auf eta^2 - kalibriert gegen die Clusterzahl"""
    rng=np.random.default_rng(seed)
    obs=eta_squared(labels,y); y=np.asarray(y,dtype=np.float64)
    ge=0
    for _ in range(n_perm):
        if eta_squared(labels,rng.permutation(y))>=obs: ge+=1
    return obs,(1.0+ge)/(1.0+n_perm)
def cluster_table(labels,y,extra=None):
    """je Cluster: Groesse, mittlere Kipprate, optionale Zusatzspalten"""
    out=[]
    for li in np.unique(labels):
        m=np.asarray(labels)==li
        row=dict(label=int(li),n=int(m.sum()),rate=float(np.mean(np.asarray(y)[m])))
        if extra:
            for k,v in extra.items(): row[k]=float(np.mean(np.asarray(v)[m]))
        out.append(row)
    return sorted(out,key=lambda r:-r["rate"])
def stability(pairs):
    """aus (eta,p) je Saat: konservativ zusammengefasst - Median-eta, schlechtestes p"""
    es=sorted(x[0] for x in pairs); ps=[x[1] for x in pairs]
    med=es[len(es)//2] if len(es)%2 else (es[len(es)//2-1]+es[len(es)//2])/2
    return dict(eta=med,eta_lo=es[0],eta_hi=es[-1],p=max(ps),spread=es[-1]-es[0])
def verdict_rdx(p_rdx,eta_rdx,eta_r0,eta_r1,alpha=0.05,margin=1.15):
    """RDX-ZEUGE nur, wenn die Differenzstruktur signifikant ist UND beide
       Einzelschichten schlaegt - sonst steckt die Struktur schon in einer
       Repraesentation und die Differenz erklaert nichts Eigenes."""
    if p_rdx>=alpha: return "KEIN-ZEUGE"
    if eta_rdx>margin*max(eta_r0,eta_r1): return "RDX-ZEUGE"
    return "NICHT-DIFFERENZIELL"
# ---------------- Klassifikator --------------------------------------------
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),(0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will would can it on as at be by".split())
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def classify_answer(t):
    if not t.strip(): return "empty"
    al=[ch for ch in t if ch.isalpha()]
    fo=[ch for ch in al if ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW)]
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
SW=("takeover","gloss","latin-switch(fr)")
KOED=["local name","native name","original name","local language","own language",
      "romaji","kanji","hiragana","katakana","transliterat","native script",
      "local script","in japanese","in chinese","in korean","original language",
      "mother tongue","local term","native term"]
def has_koeder(p):
    pl=p.lower(); return float(any(k in pl for k in KOED))
# ---------------- Quervergleich gegen den Upstream-Code ---------------------
def upstream_crosscheck(X0,X1,beta,gamma,exact,mimic):
    """rechnet denselben Graphen mit dem ORIGINALCODE. Zwei Zahlen: gegen
       unsere stabile Rechnung (da darf Praezision abweichen) und gegen die
       praezisionsgleiche Nachbildung (da muss es null sein - sonst Portierungsfehler)."""
    import subprocess, textwrap
    d="/content/rdx_upstream"
    if not os.path.isdir(os.path.join(d,"src")):
        r=subprocess.run(["git","clone","--depth","1","https://github.com/Erikiss/RDX",d],
                         capture_output=True,text=True,timeout=600)
        if r.returncode!=0: return None,None,"Klon fehlgeschlagen (privates Repo?)"
    p=os.path.join(d,"src","rdx.py")
    if not os.path.exists(p): return None,None,"src/rdx.py nicht gefunden"
    src=open(p,encoding="utf-8").read()
    def grab(name):
        i=src.find("    def %s("%name)
        if i<0: return None
        ends=[x for x in (src.find("\n    @staticmethod",i),src.find("\n    def ",i+10)) if x>0]
        return textwrap.dedent(src[i:(min(ends) if ends else len(src))])
    parts=[grab(n) for n in ("apply_guid_labels","apply_diff_function","construct_graph")]
    if any(x is None for x in parts): return None,None,"Funktionen nicht extrahierbar"
    g={"torch":torch,"np":np}
    exec("\n".join(parts),g)
    class _R: pass
    _R.apply_guid_labels=staticmethod(g["apply_guid_labels"])
    _R.apply_diff_function=staticmethod(g["apply_diff_function"])
    g["RDX"]=_R
    out=g["construct_graph"](X0,X1,dict(sim_function="neighborhood",guidance=None,
        diff_function="locally_biased",symmetrize_dm="mean",beta=beta,gamma=gamma,
        normalize_diff_mat_by_abs_max=False))
    K=("am_10","am_01","diff_10","diff_01")
    de=max(float(np.abs(out[k].numpy()-exact[k]).max()) for k in K)
    dm=max(float(np.abs(out[k].numpy()-mimic[k]).max()) for k in K)
    return de,dm,"ok"
# ---------------- Prompt-Auswahl: Verhaltens-Prompts erzwungen dazu ---------
FAM={}; BEH=set()
if META_JL and os.path.exists(META_JL):
    with open(META_JL,encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if not line: continue
            r=json.loads(line)
            pid=str(r.get("prompt_id","")).split("/")[0]
            if not pid: continue
            FAM.setdefault(pid,str(r.get("behavior_id","?")))
            if r.get("behavior_id")=="language-switching-english": BEH.add(pid)
print("Meta: %s | %d Prompts mit Verhaltensetikett | %d davon language-switching-english"
      %("gefunden" if FAM else "NICHT gefunden (Familientest entfaellt)",len(FAM),len(BEH)))
rng=np.random.default_rng(SEED)
cand=[p for p in PROMPT_IDS if 0<len(PROMPTS[p])<=MAXCHARS]
forced=[p for p in cand if p in BEH]
rest=[p for p in cand if p not in BEH]
sel=forced+[rest[i] for i in rng.permutation(len(rest))[:max(0,N_PROMPTS-len(forced))]]
FAMS=np.array([FAM.get(p,"?") for p in sel])
print("Auswahl: %d Prompts (%d aus dem Zielverhalten erzwungen, %d Familien)"
      %(len(sel),len(forced),len(set(FAMS))))
if len(forced)==0:
    print("  !! kein Verhaltens-Prompt in der Auswahl - die Zielgroesse hat"
          " vermutlich wenig Spanne.")
# ---------------- Zielgroesse: kontinuierlicher Druck, kein Boden -----------
assert MASK_NPZ and os.path.exists(MASK_NPZ), "vocab_foreign_masks.npz nicht gefunden"
_z=np.load(MASK_NPZ); M_script=torch.tensor(_z["script"])
def _mask(V):
    m=M_script.to(model.device)
    if m.shape[0]<V: m=torch.cat([m,torch.zeros(V-m.shape[0],dtype=torch.bool,device=m.device)])
    return m[:V]
NL=model.config.num_hidden_layers
assert L_LATE<NL, "L_LATE ausserhalb des Modells"
@torch.no_grad()
def repr_pair(u):
    ids=tokenizer(think_prefix(u),return_tensors="pt").input_ids.to(model.device)
    hs=model(input_ids=ids,output_hidden_states=True).hidden_states
    return (hs[L_EARLY+1][0,-1].float().cpu().numpy(),
            hs[L_LATE +1][0,-1].float().cpu().numpy())
@torch.no_grad()
def press_mass(u):
    """groesste Fremdschrift-Masse in den ersten NPOS Antwortpositionen bei
       gieriger Dekodierung. Kontinuierlich, kein Boden - anders als die
       Kipprate misst das auch dort noch, wo nie wirklich gekippt wird."""
    ids=tokenizer(think_prefix(u),return_tensors="pt").input_ids.to(model.device)
    o=model.generate(ids,do_sample=False,max_new_tokens=NPOS,
                     return_dict_in_generate=True,output_scores=True,
                     pad_token_id=tokenizer.eos_token_id)
    vals=[]
    for sc in o.scores:
        pr=torch.softmax(sc[0].float(),-1)
        vals.append(float(pr[_mask(pr.shape[0])].sum()))
    return max(vals)
@torch.no_grad()
def switch_rate(u):
    ids=tokenizer(think_prefix(u),return_tensors="pt").input_ids.to(model.device)
    o=model.generate(ids,do_sample=True,temperature=1.0,top_p=1.0,top_k=0,
                     repetition_penalty=1.0,max_new_tokens=MAX_NEW,
                     num_return_sequences=K_SAMP,pad_token_id=tokenizer.eos_token_id)
    cls=[classify_answer(tokenizer.decode(x[ids.shape[1]:],skip_special_tokens=True)) for x in o]
    return sum(1 for c in cls if c in SW)/float(K_SAMP)
E=[];L=[];YM=[];YR=[]
for i,pid in enumerate(sel):
    u=PROMPTS[pid]
    a,b=repr_pair(u); E.append(a); L.append(b)
    YM.append(press_mass(u)); YR.append(switch_rate(u))
    if (i+1)%20==0: print("  %3d/%d | Median-Druck %.2e | Kipprate %.3f"
                          %(i+1,len(sel),float(np.median(YM)),float(np.mean(YR))))
E=np.stack(E); L=np.stack(L)
YM=np.array(YM); YR=np.array(YR); Y=np.log10(np.maximum(YM,1e-12))
print("  Druck: Median %.2e | Spanne %.2e .. %.2e | log10-Streuung %.2f"
      %(float(np.median(YM)),float(YM.min()),float(YM.max()),float(Y.std())))
print("  Kipprate (nur beschreibend): Mittel %.3f | Prompts mit Kipp %d/%d"
      %(YR.mean(),int((YR>0).sum()),len(YR)))
if Y.std()<0.2:
    print("  !! WARNUNG: die Zielgroesse streut kaum (%.3f Dekaden) - unterversorgt."%Y.std())
# ---------------- RDX ------------------------------------------------------
G  =rdx_graph(E,L,beta=BETA,gamma_scale=GAMMA_SCALE,mode="stabil")
GM =rdx_graph(E,L,beta=BETA,gamma=G["gamma"],mode="torch32")
rdis=float((G["r1_rank"]!=GM["r1_rank"]).mean())
print("\nRDX-Graph (beta=%.1f, gamma=%.4f). Rangabweichung stabil gegen float32-Gram: %.2f%%"
      %(BETA,G["gamma"],100*rdis))
de,dm,msg=(None,None,"uebersprungen")
try: de,dm,msg=upstream_crosscheck(E,L,BETA,G["gamma"],G,GM)
except Exception as e: msg="Fehler: %s: %s"%(type(e).__name__,str(e)[:60])
if dm is not None:
    print("  Upstream: %.2e gegen unsere stabile Rechnung | %.2e bei gleicher Arithmetik %s"
          %(de,dm,"(Portierung bestaetigt)" if dm<1e-5 else "(!! PORTIERUNGSFEHLER)"))
else:
    print("  Quervergleich mit dem Originalcode: %s"%msg)
KTOT=N_CLUSTERS+int(ADD_NULL)
MAKE={"RDX 10 (spaet neu gruppiert)":lambda s: rdx_cluster(G["am_10"],N_CLUSTERS,FILTER_THRESH,s,ADD_NULL)[0],
      "RDX 01 (spaet getrennt)":     lambda s: rdx_cluster(G["am_01"],N_CLUSTERS,FILTER_THRESH,s,ADD_NULL)[0],
      "nur Layer %d"%L_EARLY:        lambda s: kmeans_labels(E,KTOT,s),
      "nur Layer %d"%L_LATE:         lambda s: kmeans_labels(L,KTOT,s)}
NMS=list(MAKE); SEEDS=[SEED,SEED+1,SEED+2]
HAVE_FAM=len(set(FAMS))>1
print("\nErklaerte Varianz des log-Drucks (eta^2), %d Permutationen, %d Saaten;"%(N_PERM,len(SEEDS)))
print("berichtet: Median-eta^2 und das SCHLECHTESTE p%s"
      %(" (global | innerhalb der Verhaltensfamilie)" if HAVE_FAM else ""))
STAT={}; LABS={}
for name in NMS:
    pg=[];pw=[]
    for s in SEEDS:
        lb=MAKE[name](s)
        pg.append(perm_p(lb,Y,N_PERM,SEED))
        if HAVE_FAM: pw.append(perm_p_within(lb,Y,FAMS,N_PERM,SEED))
        if s==SEED: LABS[name]=lb
    st=stability(pg); st["p_within"]=stability(pw)["p"] if HAVE_FAM else float("nan")
    STAT[name]=st
    print("  %-28s eta^2=%.4f [%.4f..%.4f]  p=%.4f%s  (%d Cluster)"
          %(name,st["eta"],st["eta_lo"],st["eta_hi"],st["p"],
            (" | %.4f"%st["p_within"]) if HAVE_FAM else "",len(np.unique(LABS[name]))))
lab10=LABS["RDX 10 (spaet neu gruppiert)"]
eta10=STAT["RDX 10 (spaet neu gruppiert)"]["eta"]
p10=STAT["RDX 10 (spaet neu gruppiert)"]["p"]
p10w=STAT["RDX 10 (spaet neu gruppiert)"]["p_within"]
eta_r0=STAT["nur Layer %d"%L_EARLY]["eta"]; eta_r1=STAT["nur Layer %d"%L_LATE]["eta"]
if STAT["RDX 10 (spaet neu gruppiert)"]["spread"]>0.5*max(eta10,1e-9):
    print("  !! saatabhaengige Partition (Spanne %.3f) - Verdikt auf wackligem Grund."
          %STAT["RDX 10 (spaet neu gruppiert)"]["spread"])
# ---------------- Was sagt der Zeuge aus? ----------------------------------
LENS=np.array([len(PROMPTS[p]) for p in sel],dtype=float)
KOE =np.array([has_koeder(PROMPTS[p]) for p in sel])
ZIEL=np.array([1.0 if p in BEH else 0.0 for p in sel])
tab=cluster_table(lab10,Y,extra={"druck":YM,"kipp":YR,"laenge":LENS,"koeder":KOE,"ziel":ZIEL})
print("\nRDX-Cluster der Richtung 10 (was Layer %d..%d neu zusammengelegt hat):"%(L_EARLY,L_LATE))
print("  %-7s %4s %10s %10s %7s %8s %7s %6s"
      %("Cluster","n","log Druck","Druck","Kipp","Zeichen","Koeder","Ziel"))
for r in tab:
    print("  %-7d %4d %10.2f %10.2e %7.3f %8.0f %7.2f %6.2f"
          %(r["label"],r["n"],r["rate"],r["druck"],r["kipp"],r["laenge"],r["koeder"],r["ziel"]))
for nm,cl in (("hoechster Druck",tab[0]["label"]),("niedrigster",tab[-1]["label"])):
    idx=[i for i in range(len(sel)) if lab10[i]==cl][:3]
    print("\n  %s (Cluster %d, log Druck %.2f) - Beispiele:"
          %(nm,cl,float(np.mean(Y[lab10==cl]))))
    for i in idx:
        t=" ".join(PROMPTS[sel[i]].split())
        print("    [%.2f | %s] %s"%(Y[i],FAMS[i][:22],t[:80]+("…" if len(t)>80 else "")))
# ---------------- Karten ----------------------------------------------------
try:
    from sklearn.manifold import SpectralEmbedding
    emb=SpectralEmbedding(n_components=2,affinity="precomputed",random_state=SEED
                          ).fit_transform(np.maximum((G["am_10"]+G["am_10"].T)/2,0))
except Exception:
    emb=None
fig=plt.figure(figsize=(15,4.6)); gs=fig.add_gridspec(1,3,wspace=.3)
ax=fig.add_subplot(gs[0,0])
if emb is not None:
    sc=ax.scatter(emb[:,0],emb[:,1],c=Y,cmap="viridis",s=42,edgecolor="#222",linewidth=.4)
    plt.colorbar(sc,ax=ax,fraction=.046,label="log10 Fremdschrift-Druck")
    if ZIEL.sum():
        ax.scatter(emb[ZIEL>0,0],emb[ZIEL>0,1],s=150,facecolors="none",
                   edgecolors="#DC2626",linewidth=1.6,label="Zielverhalten")
        ax.legend(frameon=False,fontsize=8,loc="best")
else:
    ax.text(.5,.5,"Einbettung nicht verfuegbar",ha="center")
ax.set_title("RDX-Differenzgraph (Richtung 10)\nFarbe = Druck, Ring = language-switching",fontsize=10)
ax.set_xlabel("Spektralkoordinate 1"); ax.set_ylabel("Spektralkoordinate 2")
ax2=fig.add_subplot(gs[0,1])
cls=[r["label"] for r in tab]; rts=[r["rate"] for r in tab]
ses=[float(np.std(Y[lab10==c],ddof=1)/max(np.sqrt((lab10==c).sum()),1)) if (lab10==c).sum()>1
     else 0.0 for c in cls]
ax2.bar(range(len(cls)),rts,yerr=[1.96*s for s in ses],capsize=4,
        color=["#DC2626" if r>Y.mean() else "#2563EB" for r in rts])
ax2.axhline(Y.mean(),ls="--",c="#666",lw=1,label="Gesamtmittel %.2f"%Y.mean())
ax2.set_xticks(range(len(cls))); ax2.set_xticklabels(["C%d\nn=%d"%(c,r["n"]) for c,r in zip(cls,tab)],fontsize=8)
ax2.set_ylabel("log10 Druck"); ax2.set_title("Druck je RDX-Cluster (95 %)",fontsize=10)
ax2.legend(frameon=False,fontsize=8)
ax3=fig.add_subplot(gs[0,2])
nms=NMS; es=[STAT[n]["eta"] for n in nms]
pv=[(STAT[n]["p_within"] if HAVE_FAM else STAT[n]["p"]) for n in nms]
err=[[STAT[n]["eta"]-STAT[n]["eta_lo"] for n in nms],[STAT[n]["eta_hi"]-STAT[n]["eta"] for n in nms]]
ax3.barh(range(len(nms))[::-1],es,xerr=err,capsize=3,
         color=["#DC2626","#F59E0B","#9CA3AF","#9CA3AF"])
for i,(e,p) in enumerate(zip(es,pv)):
    ax3.text(STAT[nms[i]]["eta_hi"]+.006,len(nms)-1-i,"p=%.3f"%p,va="center",fontsize=8)
ax3.set_yticks(range(len(nms))[::-1]); ax3.set_yticklabels(nms,fontsize=8)
ax3.set_xlabel("eta² (erklaerter log-Druck)")
ax3.set_title("Differenzstruktur gegen Einzelschichten\n(Balken: Median, Fehler: Saatspanne)",fontsize=9)
ax3.set_xlim(0,max([STAT[n]["eta_hi"] for n in nms])*1.4+.02)
plt.show()
# ---------------- Verdikt ---------------------------------------------------
code=verdict_rdx(p10,eta10,eta_r0,eta_r1)
print("\nVERDIKT:",end=" ")
if code=="RDX-ZEUGE":
    print("RDX-ZEUGE STEHT: die unbeaufsichtigte Differenzstruktur zwischen Layer %d"%L_EARLY)
    print("  und %d erklaert den Fremdschrift-Druck (eta^2=%.3f, p=%.4f)"%(L_LATE,eta10,p10))
    print("  und schlaegt beide Einzelschichten (%.3f / %.3f). Der Zeuge steht in der"%(eta_r0,eta_r1))
    print("  Repraesentationsdifferenz, nicht im Text: das Modell schreibt ihn nicht,")
    print("  also kann es ihn auch nicht abkoppeln. Ihn zu eliminieren hiesse, die")
    print("  Differenz aufzuloesen - und das ist genau die Verhaltensaenderung.")
elif code=="NICHT-DIFFERENZIELL":
    print("ZEUGE JA, DIFFERENZ NEIN: die Partition erklaert den Druck (eta^2=%.3f,"%eta10)
    print("  p=%.4f), aber eine einzelne Schicht tut es genauso gut"%p10)
    print("  (%.3f / %.3f). Die Disposition ist unbeaufsichtigt ablesbar - der Zeuge"%(eta_r0,eta_r1))
    print("  steht und ist nicht abkoppelbar. Nur braucht es dafuer kein RDX; die")
    print("  Repraesentation selbst genuegt als Protokoll.")
else:
    pr0=STAT["nur Layer %d"%L_EARLY]; pr1=STAT["nur Layer %d"%L_LATE]
    q0=pr0["p"]; q1=pr1["p"]
    if min(q0,q1)<0.05:
        print("ZEUGE JA, ABER NICHT IN DER DIFFERENZ: der RDX-Graph erklaert nichts")
        print("  (eta^2=%.3f, p=%.4f), eine EINZELNE Schicht dagegen schon"%(eta10,p10))
        print("  (Layer %d: eta^2=%.3f p=%.4f | Layer %d: eta^2=%.3f p=%.4f)."
              %(L_EARLY,eta_r0,q0,L_LATE,eta_r1,q1))
        print("  Der Zeuge steht - aber als Repraesentation, nicht als Differenz.")
    else:
        print("KEIN ZEUGE AN DIESER STELLE: keine Partition erklaert den Druck")
        print("  (RDX eta^2=%.3f, p=%.4f; Einzelschichten %.3f / %.3f)."%(eta10,p10,eta_r0,eta_r1))
        print("  Am letzten Prompt-Token ist die Disposition nicht clusterlesbar - was")
        print("  zu Cell 19 passt. Naechster Ansatz: Koeder-Position 43 statt letztes")
        print("  Token, oder KV-Zustaende statt Residuum.")
if HAVE_FAM:
    print("\nSpezifitaet: mischt man den Druck nur INNERHALB der Verhaltensfamilien,")
    print("  bleibt p=%.4f."%p10w)
    if p10w<0.05:
        print("  Die Partition traegt auch dann - sie bildet nicht bloss die Katalog-")
        print("  Familie nach, sondern trennt auch innerhalb einer Familie.")
    else:
        print("  Der Effekt verschwindet: die Partition trennt im Wesentlichen die")
        print("  Katalog-Familien, nicht Prompts innerhalb einer Familie. Das schwaecht")
        print("  den Befund - hebt ihn aber nicht auf, denn Familie und Druck sind hier")
        print("  fast kollinear (nur %d Prompts des Zielverhaltens im Pool)."%int(ZIEL.sum()))
print("(Zielgroesse ist der Fremdschrift-Druck an den ersten %d Antwortpositionen,"%NPOS)
print(" nicht die Kipprate - die hat bei gemischten Prompts praktisch keine Spanne.)")
RDX_RESULTS=dict(verdict=code,stats=dict((n,STAT[n]) for n in nms),n=len(sel),
                 n_ziel=int(ZIEL.sum()),rank_disagree=rdis,clusters=tab,
                 upstream=dict(exact=de,mimic=dm),layers=(L_EARLY,L_LATE),p_within=p10w)
